# Tensor Decomposition
*Thomas Dooms*

In tutorial 1, we decomposed the bilinear interaction tensor using eigendecomposition. This gives us interpretable features *per output class*, but there are some issues with this. The most important one is orthogonality; you might have overlapping structures in input space which behave differently. The second is somewhat of a consequence where eigendecompositions often yield "superposed" features; we might hope that a 6 decomposes into a few edge-detectors but that generally doesn't happen. If the model shares structure across classes, eigendecomposition can't see it.

Here we take a different approach. Instead of decomposing per class, we decompose the *full* third-order tensor into a set of shared **neurons**, each with explicit input patterns and output weights. The result is a set of interpretable building blocks that the model combines to classify digits.

### Setup

In [1]:
!git clone https://github.com/shengweiming/bilinear-decomposition

Cloning into 'bilinear-decomposition'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 144 (delta 50), reused 113 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 17.50 MiB | 18.30 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [2]:
%cd bilinear-decomposition

/content/bilinear-decomposition


In [3]:
!pip install -e .

Obtaining file:///content/bilinear-decomposition
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 9.4 MB/s eta 0:00:00
  Building editable for bilinear-decom

In [ ]:
!pip install -U ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.3/627.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [1]:
import torch
import plotly.express as px
from plotly.subplots import make_subplots

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

device = "cuda:0"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.72MB/s]


### Training the original model

As in tutorial 1, we train a bilinear model with Gaussian noise augmentation to prevent overfitting on individual pixels.

In [2]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

train/loss: 0.145, train/acc: 0.958, val/loss: 0.110, val/acc: 0.969: 100%|██████████| 20/20 [00:11<00:00,  1.68it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.132581,0.687517,0.446589,0.8836
1,0.453151,0.872559,0.334491,0.9104
2,0.386020,0.891669,0.282399,0.9237
3,0.343214,0.901451,0.247095,0.9311
4,0.307735,0.913423,0.206934,0.9413
5,0.259505,0.925394,0.183544,0.9487
6,0.231982,0.931944,0.161961,0.9553
7,0.219108,0.936742,0.148345,0.9598
8,0.197924,0.942854,0.137090,0.9614
9,0.185841,0.946356,0.131954,0.9627


### From eigendecomposition to tensor decomposition

Recall that the bilinear model computes $\text{output}_c = \sum_{i,j} B_{c,i,j}\, x_i\, x_j$, where $B$ is the third-order interaction tensor. Eigendecomposition slices $B$ along the class axis and decomposes each slice independently.

We can instead factorize the *entire* tensor at once into a new bilinear layer:
$$B_{c,i,j} \approx \sum_{r=1}^{R} L_{i,r}\, R_{j,r}\, D_{c,r}$$

Each component $r$ is a **neuron** with three parts:
- $L_r \in \mathbb{R}^{784}$, the left input pattern
- $R_r \in \mathbb{R}^{784}$, the right input pattern  
- $D_r \in \mathbb{R}^{10}$, the output weights over classes

The neuron's activation on input $x$ is $(L_r^\top x)(R_r^\top x)$. Using the identity $ab = \tfrac{1}{4}(a+b)^2 - \tfrac{1}{4}(a-b)^2$, this becomes:
$$\tfrac{1}{4}\big((L_r + R_r)^\top x\big)^2 - \tfrac{1}{4}\big((L_r - R_r)^\top x\big)^2$$

So each neuron naturally decomposes into a **positive pattern** $L_r + R_r$ (matching it increases activation) and a **negative pattern** $L_r - R_r$ (matching it decreases activation). These are directly analogous to eigenvectors with positive and negative eigenvalues.

### Fitting the decomposition

We fit a new bilinear layer by maximizing the cosine similarity between the original interaction tensor and our approximation. The `Sparse` model stores $L$, $R$, and $D$ as learnable parameters and is optimized with Muon.

In [3]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = 1 - sparse.similarity(model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

loss: 0.0019: 100%|██████████| 200/200 [00:02<00:00, 72.88it/s]

Original: 0.969, Sparse: 0.968


### Visualizing neurons

Each neuron has a positive pattern ($L_r + R_r$), a negative pattern ($L_r - R_r$), and output weights ($D_r$). The `decompose` method returns these normalized and sorted by importance (the product of the factor norms).

In [4]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [5]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Exercises

The goal of this part is to implement a decomposition that is more in line with our requirements. This is where you try to find what kinds of priors and structure work well. The code above provides a skeleton with the essentials: the `Sparse` model, the reconstruction loop, and the visualization. There's no right answer and this is a hard task to get right.